## LeNet5 — Torch + LiteRT build pipeline

Trains a LeNet5 on MNIST, exports an int8 TFLite model via post-training quantization (analogous to the TensorFlow lenet notebook), using `litert-torch`.

Outputs

- `models/lenet5_quantized_torch.tflite` INT8 input/output, per-tensor, NHWC; MicroFlow-compatible

Keep `epochs` small if you only need artifacts quickly.

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

SEED = 3407
torch.manual_seed(SEED)

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE_QUANT = MODELS_DIR / "lenet5_quantized_torch.tflite"

DEVICE = "cpu"
print("Model output:", OUT_TFLITE_QUANT)
print("Device:", DEVICE)

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


transform = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=str(REPO_ROOT / "dataset"), train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

model = LeNet5().to(DEVICE)
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(dim=1)
            correct += int((pred == yb).sum().item())
            total += yb.size(0)
    return correct / max(total, 1)


EPOCHS = 2
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in tqdm(train_loader, desc=f"epoch {epoch + 1}"):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    val_acc = evaluate(test_loader)
    print(f"epoch={epoch + 1} val_acc={val_acc:.4f}")

print("Test accuracy:", evaluate(test_loader))

In [ ]:
import litert_torch
from ai_edge_litert import interpreter as tfl_interpreter
import tensorflow as tf

model.eval()

# Wrap model to accept NHWC
nhwc_model = litert_torch.to_channel_last_io(model, args=[0])
sample_input = torch.zeros(1, 28, 28, 1)  # NHWC

calib_loader = DataLoader(test_ds, batch_size=1, shuffle=False)


def representative_data_gen():
    for i, (xb, _) in enumerate(calib_loader):
        if i >= 200:
            break
        # dataloader gives NCHW; convert to NHWC for calibration
        yield [xb.permute(0, 2, 3, 1).numpy()]


tfl_flags = {
    "optimizations": [tf.lite.Optimize.DEFAULT],
    "representative_dataset": representative_data_gen,
    "target_spec": {
        "supported_ops": [tf.lite.OpsSet.TFLITE_BUILTINS_INT8],
        "supported_types": [tf.int8],
    },
    "inference_input_type": tf.int8,
    "inference_output_type": tf.int8,
    "_experimental_disable_per_channel": True,
    "_experimental_disable_per_channel_quantization": True,
}

edge_model = litert_torch.convert(
    nhwc_model, (sample_input,), _ai_edge_converter_flags=tfl_flags
)

edge_model.export(str(OUT_TFLITE_QUANT))
print("Wrote int8 per-tensor TFLite to", OUT_TFLITE_QUANT)

interp = tfl_interpreter.Interpreter(model_content=edge_model.tflite_model())
interp.allocate_tensors()
in_info = interp.get_input_details()[0]
out_info = interp.get_output_details()[0]
print("input: ", in_info["dtype"], in_info["shape"], "quant=", in_info["quantization"])
print("output:", out_info["dtype"], out_info["shape"], "quant=", out_info["quantization"])
